# Notebook for correlation and association plots
Needs to be run after `preprocessing.ipynb`

In [ ]:
from __future__ import annotations

import itertools
import typing
from pathlib import Path

import pandas as pd

from utils.corr_assoc_utils import create_corr_assoc_matrix
from utils.misc_utils import turn_cats_to_continuous
from utils.plotting_utils import make_plot_corr_assoc_matrix
from utils.saving_utils import load_data, save_plot

pd.options.mode.copy_on_write = True

In [ ]:
# Read in data
data_reduced_path = Path("data/preprocessed/data_discov_reduced.pkl")
data_full_path = Path("data/preprocessed/data_discov_full.pkl")
vars_info_path = Path("data/preprocessed/vars_info_full.pkl")

data_full: pd.DataFrame = load_data(data_full_path)
data_reduced: pd.DataFrame = load_data(data_reduced_path)
vars_info: pd.DataFrame = load_data(vars_info_path)

# Set outputs folder
outputs_folder = "outputs/corr_assoc_plots"

In [ ]:
# Get variable groups
var_groups0 = {group: vars_info.loc[vars_info["group"] == group, "name"] for group in vars_info["group"].cat.categories}
var_groups = typing.cast(dict[str, list[str]], var_groups0)

In [ ]:
# Select variables from reduced data
groups = ["alcohol"]  # Change to one or more of: "alcohol", "nicotine", "other_drugs", "medication"
vars_group = itertools.chain.from_iterable(var_groups[group] for group in groups)
vars_group = [var for var in data_reduced.columns if var in vars_group]  # Only keep those that occur in reduced data

# Add vars from full data
full_data_groups = ["general", "socioecon"]
vars_plot = [*vars_group, *itertools.chain.from_iterable(var_groups[group] for group in full_data_groups)]
vars_plot = [var for var in vars_plot if var not in ["fall", "zeit"]]  # Remove uninteresting variables

In [ ]:
# Treat binary and ordinal variables as if they were continuous (with equally spaced category coding)
treat_cat_as_continuous = vars_info.loc[vars_info["type"].isin(["binary", "ordinal"]), "name"]
data_plot = turn_cats_to_continuous(data_full[vars_plot], treat_cat_as_continuous)

In [ ]:
# corr_assoc_matrix, p_value_matrix = create_corr_assoc_matrix(data_plot)
# corr_assoc_matrix

In [ ]:
plot_corr_assoc_matrix = make_plot_corr_assoc_matrix(data_plot)
plot_corr_assoc_matrix.show()

save_plot(plot_corr_assoc_matrix, Path(outputs_folder, "plot_corr_assoc_matrix_" + "_".join(groups) + ".png"), dpi=150)